# TrustBench — AI vs US-population (and WVS) comparison

Tied to the AI-legitimacy / institutional-trust safety framing. Output figures in `figures/us_comparison/`, narrative in `REPORT.md` next to them.

**Note.** WVS-7 did *not* administer Q292 (politicians) to US respondents — the AI-vs-US politicians figure cannot be built from this dataset, so Figure 3 falls back to AI vs WVS-13-country pool.

In [ ]:
from __future__ import annotations
import sys, pathlib
REPO = pathlib.Path.cwd()
while not (REPO / 'data').exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from src.analysis.loader   import load_items, load_llm_results, MODEL_ORDER
from src.analysis.scoring  import add_trust_score
from src.analysis.aggregate import model_item_means
from src.analysis.us_comparison import (
    us_item_means, profile_similarity, shortlist_item_order,
    SHORTLIST_INSTITUTIONS, POLITICAL_BUCKETS,
)
from src.analysis.us_figures import make_all
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)

In [ ]:
items = load_items()
llm   = load_llm_results('all')
scored = add_trust_score(llm, items)
print('AI rows :', len(scored), '  refusal rate %:',
      round(scored['refused'].mean() * 100, 2))

## 1 · US-population trust (all + Liberal / Centrist / Conservative)

In [ ]:
us_all = us_item_means(items, political_split=False).set_index('item_id')
us_pol = us_item_means(items, political_split=True)
print('US (all) — sample sizes per item (first 5):')
print(us_all[['n']].head())
print()
print('US political-split sample sizes (mean across items):')
print(us_pol.groupby('group')['n'].mean().round(0))

## 2 · AI vs US gap on the 12-institution shortlist

In [ ]:
short_ids = shortlist_item_order(items)
items_lab = items.set_index('id')
ai = model_item_means(scored, by=['model','item_id']).pivot(
    index='item_id', columns='model', values='trust_mean').reindex(short_ids)[MODEL_ORDER]
tab = ai.copy()
tab.insert(0, 'US (all)', us_all['trust_mean'].reindex(short_ids))
delta = ai.sub(us_all['trust_mean'].reindex(short_ids), axis=0)
tab['max |AI−US|'] = pd.Series(np.nanmax(np.abs(delta.to_numpy()), axis=1), index=delta.index)
tab.index = [items_lab.loc[i,'institution'] for i in short_ids]
tab.round(3)

## 3 · US Liberal vs Conservative gaps — and where AI lands

In [ ]:
us_pol_wide = us_pol.pivot(index='item_id', columns='group', values='trust_mean')
gap = (us_pol_wide['Liberal (Q240 1-4)'] - us_pol_wide['Conservative (Q240 6-10)'])
tab2 = pd.DataFrame({
    'US-Liberal':       us_pol_wide['Liberal (Q240 1-4)'].reindex(short_ids),
    'US-Conservative':  us_pol_wide['Conservative (Q240 6-10)'].reindex(short_ids),
    'Lib − Cons':       gap.reindex(short_ids),
}).join(ai)
tab2.index = [items_lab.loc[i,'institution'] for i in short_ids]
tab2.reindex(tab2['Lib − Cons'].abs().sort_values(ascending=False).index).round(3)

## 4 · Profile similarity: each AI vs every reference (Spearman ρ on Q64–Q89)

In [ ]:
ps = profile_similarity(scored, items, section='wvs_confidence')
us_view = ps[ps['ref_kind'].isin(['US-aggregate','US-political'])].pivot(
    index='ref_label', columns='model', values='spearman').round(3)
us_view

In [ ]:
# Top-5 country matches per model
for m in MODEL_ORDER:
    top = (ps[(ps['model']==m) & (ps['ref_kind']=='WVS-country')]
           .sort_values('spearman', ascending=False).head(5))
    print(f'\n{m}:')
    print(top[['ref_label','spearman','rmse']].to_string(index=False))

## 5 · Generate the four publication figures

In [ ]:
make_all(scored, items)
import os
for f in sorted(os.listdir(REPO / 'figures' / 'us_comparison')):
    p = REPO / 'figures' / 'us_comparison' / f
    print(f'  {f}  {p.stat().st_size // 1024} KB')

---
Findings keyed to the safety/legitimacy framing live in [`figures/us_comparison/REPORT.md`](../../figures/us_comparison/REPORT.md).